In [ ]:
from databricks import sql
import os
from dotenv import load_dotenv
from grocery_list import get_grocery_list
from product_info import get_products_info
import pandas as pd
from databricks_info import sql_commands

load_dotenv()

headers = {
	"x-rapidapi-key": os.getenv("RAPIDAPI_KEY"),
	"x-rapidapi-host": "woolworths-products-api.p.rapidapi.com",
	"Content-Type": "application/json"
}

groceries = get_products_info(get_grocery_list(), headers)
my_groceries = [grocery.get('results')[0] for grocery in groceries]

In [2]:

from datetime import datetime

connection = sql.connect(
    server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME"),
    http_path = os.getenv("DATABRICKS_HTTP_PATH"),
    access_token = os.getenv("DATABRICKS_ACCESS_TOKEN")
)

cursor = connection.cursor()
cursor.execute("CREATE DATABASE IF NOT EXISTS woolis")
cursor.execute("USE woolis")
df = pd.DataFrame(my_groceries)
retracted_date = datetime.now().strftime("%Y-%m-%d")
retracted_time = datetime.now().strftime("%H:%M:%S")
df["date_retracted"] = retracted_date
df["time_retracted"] = retracted_time
#cursor.execute("DROP TABLE IF EXISTS products")
cursor.execute(sql_commands(df, 1))
for row in df.itertuples(index=False, name=None):
    cursor.execute(sql_commands(df, 2), row)
cursor.execute("SELECT * FROM products")
rows = cursor.fetchall()
df = pd.DataFrame(rows, columns=[col[0] for col in cursor.description])
print(df.to_string())
cursor.close()
connection.close()

          barcode                                         product_name product_brand current_price product_size                                                       url date_retracted time_retracted
0   9310088012385  U by Kotex Ultrathin Overnight Pads Long With Wings    U by Kotex           5.6       8 pack  https://www.woolworths.com.au/shop/productdetails/684420     2026-08-27       11:17:37
1   9300677010649                Cloverdale Pure Honey Twist & Squeeze    Cloverdale           3.3         375g   https://www.woolworths.com.au/shop/productdetails/89159     2026-08-25       21:00:18
2   9310088012385  U by Kotex Ultrathin Overnight Pads Long With Wings    U by Kotex           3.9       8 pack  https://www.woolworths.com.au/shop/productdetails/684420     2026-08-25       21:04:41
3   9310088012385  U by Kotex Ultrathin Overnight Pads Long With Wings    U by Kotex           3.9       8 pack  https://www.woolworths.com.au/shop/productdetails/684420     2026-08-25       21:00:18
